# 49 — Decoding & sampling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/049-decoding-and-sampling.ipynb)

The model gives you logits; the decoder chooses a token. In this notebook you reshape four logits with temperature, apply a top-p cutoff, and see the surviving set change as the threshold moves.

In [ ]:
import numpy as np                                         # array helpers
z = np.array([3.0, 1.0, 0.0, -1.0])                        # four candidate logits
def softmax(z, T=1.0):                                     # temperature softmax
    scaled = z / T                                         # divide logits by T
    e = np.exp(scaled - scaled.max())                      # stable exponentiation
    return e / e.sum()                                     # normalize to 1
def nucleus(probs, p):                                     # top-p surviving ids
    order = np.argsort(-probs)                             # highest probability first
    keep, total = [], 0.0                                  # survivors and mass
    for i in order:                                        # walk down the sorted ids
        keep.append(int(i))                                # keep this id
        total += probs[i]                                  # add its probability
        if total >= p - 1e-12:                             # reached the threshold
            break                                          # stop adding
    return keep                                            # ids in sorted order
for T in (0.5, 1.0, 2.0):                                  # three temperatures
    probs = softmax(z, T)                                  # reshape the distribution
    print(f"T={T} top={probs[0]:.3f} second={probs[1]:.3f}")  # compare
probs = softmax(z, 1.0)                                    # the model's own distribution
kept = nucleus(probs, 0.9)                                 # 90% nucleus
print("kept", kept, "mass", round(float(probs[kept].sum()), 3))  # show the cut

## What the numbers mean

The top probability is 0.831 at `T = 1`, 0.979 at `T = 0.5` and 0.579 at `T = 2`. Temperature changes the shape of the distribution but not its ordering: no token is added or removed. The nucleus step is what removes tokens — with `p = 0.9` the first two probabilities sum to 0.943, so only two survive and are renormalized to 0.881 and 0.119.

In [ ]:
import matplotlib.pyplot as plt                            # plotting library
temps = np.linspace(0.2, 2.0, 40)                          # temperatures to test
tops = [softmax(z, T)[0] for T in temps]                   # top probability each time
plt.figure(figsize=(6, 3))                                 # wide figure
plt.plot(temps, tops, color="tab:purple")                   # the curve
plt.axhline(0.9, color="grey", linestyle="--")              # a 90% confidence line
plt.xlabel("temperature")                                  # x label
plt.ylabel("top-token probability")                        # y label
plt.title("low T sharpens, high T flattens")                # takeaway
plt.show()                                                 # render

## Reading the curve

As temperature rises, the top token loses mass monotonically. That is why a high temperature makes rare tokens more likely but does not make the answer more knowledgeable: the model's scores are fixed, and the decoder is only flattening them.

For extraction and code, stay near the left side of this plot — greedy decoding is the limit at zero. For open-ended text, use top-p to cut the tail rather than relying on temperature alone.

## Your turn

Run the nucleus with `p = 0.5`, `0.8` and `1.0` and record the set size each time. Then take the flat distribution `[0.3, 0.25, 0.24, 0.21]` and show that top-p 0.9 keeps all four while top-k 2 keeps two. Finally, implement a seeded sampler with `np.random.default_rng(0).choice` and check that two runs with the same seed agree.